## Лабораторная работа 6: Дискретное косинусное преобразование (DCT)

### Упражнение 6.1: Основы DCT

DCT - это преобразование, похожее на DFT, но использующее только косинусные функции.

In [ ]:
import sys
sys.path.insert(0, '../ThinkDSP/code')

from thinkdsp import TriangleSignal, SquareSignal, SawtoothSignal
from thinkdsp import decorate
import matplotlib.pyplot as plt
import numpy as np
from scipy.fftpack import dct, idct

### 1. Сравнение DFT и DCT

In [ ]:
# Создание тестового сигнала
signal = TriangleSignal(freq=400)
wave = signal.make_wave(duration=0.01, framerate=10000)

# DFT
spectrum_dft = wave.make_spectrum()

# DCT
dct_coeffs = dct(wave.ys, type=2, norm='ortho')

# Визуализация
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
wave.plot()
plt.title('Исходный сигнал')
decorate(xlabel='Время (с)', ylabel='Амплитуда')

plt.subplot(1, 3, 2)
spectrum_dft.plot()
plt.title('DFT')
decorate(xlabel='Частота (Гц)', ylabel='Амплитуда')

plt.subplot(1, 3, 3)
plt.plot(np.abs(dct_coeffs[:50]))
plt.title('DCT')
decorate(xlabel='Индекс коэффициента', ylabel='Амплитуда')

plt.tight_layout()
plt.show()

**Вопрос 1:** В чем основное отличие между DFT и DCT?

### 2. Сжатие сигнала с помощью DCT

In [ ]:
def compress_dct(signal, keep_ratio=0.1):
    """
    Сжатие сигнала путем обнуления малых коэффициентов DCT
    """
    # Прямое DCT
    coeffs = dct(signal, type=2, norm='ortho')
    
    # Определение порога
    n_keep = int(len(coeffs) * keep_ratio)
    
    # Сортировка по абсолютному значению
    indices = np.argsort(np.abs(coeffs))[::-1]
    
    # Обнуление малых коэффициентов
    compressed = np.zeros_like(coeffs)
    compressed[indices[:n_keep]] = coeffs[indices[:n_keep]]
    
    # Обратное DCT
    reconstructed = idct(compressed, type=2, norm='ortho')
    
    return reconstructed, compressed

# Применение к сигналу
original = wave.ys
reconstructed, compressed = compress_dct(original, keep_ratio=0.2)

# Визуализация
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(wave.ts, original, label='Оригинал', alpha=0.7)
plt.plot(wave.ts, reconstructed, label='Восстановленный', alpha=0.7)
plt.legend()
plt.title('Сравнение сигналов')
decorate(xlabel='Время (с)', ylabel='Амплитуда')

plt.subplot(1, 2, 2)
plt.plot(np.abs(dct(original, type=2, norm='ortho')), label='Все коэффициенты', alpha=0.7)
plt.plot(np.abs(compressed), label='20% коэффициентов', alpha=0.7)
plt.legend()
plt.title('DCT коэффициенты')
decorate(xlabel='Индекс', ylabel='Амплитуда')

plt.tight_layout()
plt.show()

# Вычисление ошибки
mse = np.mean((original - reconstructed)**2)
print(f"Среднеквадратичная ошибка: {mse:.6f}")
print(f"Степень сжатия: {100 * (1 - 0.2):.0f}%")

**Вопрос 2:** Почему DCT эффективно для сжатия сигналов?

### 3. Применение к различным сигналам

In [ ]:
# Тестирование на разных сигналах
signals = [
    ('Треугольник', TriangleSignal(freq=400)),
    ('Прямоугольник', SquareSignal(freq=400)),
    ('Пила', SawtoothSignal(freq=400))
]

fig, axes = plt.subplots(3, 2, figsize=(12, 10))

for i, (name, signal) in enumerate(signals):
    wave = signal.make_wave(duration=0.01, framerate=10000)
    reconstructed, _ = compress_dct(wave.ys, keep_ratio=0.2)
    
    # Оригинал
    axes[i, 0].plot(wave.ts, wave.ys)
    axes[i, 0].set_title(f'{name} - Оригинал')
    axes[i, 0].set_xlabel('Время (с)')
    axes[i, 0].set_ylabel('Амплитуда')
    
    # Восстановленный
    axes[i, 1].plot(wave.ts, reconstructed)
    axes[i, 1].set_title(f'{name} - Восстановленный (20%)')
    axes[i, 1].set_xlabel('Время (с)')
    axes[i, 1].set_ylabel('Амплитуда')
    
    # Ошибка
    mse = np.mean((wave.ys - reconstructed)**2)
    print(f"{name}: MSE = {mse:.6f}")

plt.tight_layout()
plt.show()

**Вопрос 3:** Для каких типов сигналов DCT дает лучшее сжатие?

### Задания для самостоятельной работы

1. Исследуйте влияние степени сжатия (keep_ratio) на качество восстановленного сигнала.
2. Примените DCT к реальному аудиосигналу и оцените качество сжатия.
3. Сравните эффективность сжатия DCT и DFT.
4. Реализуйте простой алгоритм сжатия изображений на основе DCT (2D-DCT).

In [ ]:
# Место для вашего кода
